# IIIF: One Standard, Every Museum

In notebook 01 you searched Europeana and got *metadata* back. This notebook is about the **pixels**, and about how a single open standard, **IIIF** (*triple-eye-eff*), lets you ask any participating institution for *exactly the part of an image you need*, without downloading the whole file.

In this notebook you will:

1. See **why IIIF matters** (overview graphics)
2. Read a real **manifest** and find the image service inside it
3. Request **regions, sizes, rotations and colour modes** just by editing a URL
4. **Click and drag** on an image to find pixel coordinates, then attach **annotations** to them


## Setup

In [ ]:
import html
import json
from IPython.display import Image, HTML, display
import requests

# A Europeana item whose institution (Wellcome Collection) runs a real IIIF image server.
# Europeana item: https://www.europeana.eu/item/9200579/szdw46pt
MANIFEST_URL = "https://iiif.europeana.eu/presentation/9200579/szdw46pt/manifest?format=3"

# Used only if the network is unavailable, so the notebook still renders
FALLBACK = {
    "title": "Six sea horses in the sea. Colour line block after a painting by H. Murayama.",
    "canvas_id": "https://iiif.europeana.eu/presentation/9200579/szdw46pt/canvas/p1",
    "canvas_w": 512, "canvas_h": 824,
    "service": "https://iiif.wellcomecollection.org/image/V0022092.jpg",
}

## Why IIIF? The virtues at a glance

In [ ]:
CARDS = [
    ("🔌", "Interoperable", "One viewer (Mirador, Universal Viewer, …) opens objects from any compliant institution."),
    ("✂️", "Pixels on demand", "The server crops, scales, rotates and re-encodes for you. You get only what you ask for."),
    ("🔍", "Deep zoom, tiny downloads", "Gigapixel scans stay usable: viewers fetch just the visible tiles."),
    ("📌", "Annotations that point", "A note targets a <i>region of a canvas</i>. It never copies or edits the image."),
    ("🔗", "URLs are citations", "Every view of an image is a stable, shareable URL you can put in a footnote."),
    ("🧩", "Metadata travels along", "Title, rights and attribution come bundled in the manifest next to the image."),
]
cards = "".join(
    f'<div class="vc"><div class="vi">{i}</div><div class="vt">{t}</div><div class="vd">{d}</div></div>'
    for i, t, d in CARDS
)
display(HTML("""
<style>
.vg{display:grid;grid-template-columns:repeat(auto-fit,minmax(210px,1fr));gap:12px;font-family:system-ui,sans-serif}
.vc{background:#f4f7fc;border:1px solid #d5ddec;border-top:4px solid #1c7ed6;border-radius:8px;padding:12px 14px;color:#1d2433}
.vi{font-size:26px}.vt{font-weight:700;margin:4px 0}.vd{font-size:13px;line-height:1.4;color:#42506a}
</style><div class="vg">""" + cards + "</div>"))

## Get a manifest and find the image service

The **manifest** is a JSON document that describes an object. Europeana generates one for every item. Inside it, a hierarchy leads to the image:

`Manifest → Canvas → Annotation (painting) → Image body → **service**`

The `service` id is the base address for the **Image API**. That's the one string you need.

In [ ]:
def load_item():
    try:
        m = requests.get(MANIFEST_URL, timeout=20).json()
        canvas = m["items"][0]
        body = canvas["items"][0]["items"][0]["body"]
        service = body["service"][0]["id"] if isinstance(body["service"], list) else body["service"]["id"]
        title = next(iter(m["label"].values()))[0]
        return {"title": title, "canvas_id": canvas["id"], "canvas_w": canvas["width"],
                "canvas_h": canvas["height"], "service": service.rstrip("/")}
    except Exception as e:
        print(f"⚠️ Could not read live manifest ({e}); using built-in fallback.")
        return FALLBACK

item = load_item()
service = item["service"]          # <- the image service id from the canvas in the manifest

# info.json tells us the TRUE full-resolution size (the canvas may use a smaller coordinate space)
try:
    info = requests.get(f"{service}/info.json", timeout=20).json()
    FULL_W, FULL_H = info["width"], info["height"]
except Exception:
    FULL_W, FULL_H = item["canvas_w"], item["canvas_h"]

print("title        :", item["title"])
print("canvas       :", item["canvas_id"], f"({item['canvas_w']} × {item['canvas_h']})")
print("image service:", service)
print("full size    :", f"{FULL_W} × {FULL_H} px")

### The chain, drawn from the real values above

In [ ]:
def chain_graphic():
    e = html.escape
    steps = [
        ("#7048e8", "Manifest", "Presentation API", e(MANIFEST_URL.split("?")[0].replace("https://", ""))),
        ("#1c7ed6", "Canvas", "a virtual page", e(f"{item['canvas_w']} × {item['canvas_h']} units")),
        ("#0ca678", "Annotation", "motivation: painting", "image → canvas"),
        ("#e8590c", "Image service", "Image API", e(service.replace("https://", ""))),
        ("#c92a2a", "Pixels", "any region · size · rotation", f"{FULL_W} × {FULL_H} available"),
    ]
    boxes = '<div class="ar">→</div>'.join(
        f'<div class="st" style="border-top-color:{c}"><div class="sn" style="color:{c}">{n}</div>'
        f'<div class="ss">{s}</div><div class="sv">{v}</div></div>' for c, n, s, v in steps)
    return """
<style>
.ch{display:flex;align-items:stretch;gap:6px;flex-wrap:wrap;font-family:system-ui,sans-serif}
.st{flex:1;min-width:120px;background:#f4f7fc;border:1px solid #d5ddec;border-top:5px solid;border-radius:8px;padding:10px 12px;color:#1d2433}
.sn{font-weight:700;font-size:16px}.ss{font-size:12px;color:#5b6577;margin-bottom:6px}
.sv{font:11px ui-monospace,monospace;overflow-wrap:anywhere;color:#1d2433}
.ar{align-self:center;font-size:22px;color:#7a869c}
</style><div class="ch">""" + boxes + "</div>"

display(HTML(chain_graphic()))

## Anatomy of an Image API URL

```
{service}/{region}/{size}/{rotation}/{quality}.{format}
```

Every segment is optional to *think* about but mandatory to *write*. Hover a segment for its meaning.

In [ ]:
def url_anatomy(region="full", size="max", rotation="0", quality="default", fmt="jpg"):
    segs = [
        ("#868e96", service, "service: where the image lives (from the manifest)"),
        ("#1c7ed6", region, "region: full · square · x,y,w,h in pixels · pct:x,y,w,h"),
        ("#0ca678", size, "size: max · 500, (width) · ,300 (height) · pct:50 · !w,h (fit inside)"),
        ("#e8590c", rotation, "rotation: 0 · 90 · 180 · 270 · prefix ! to mirror"),
        ("#7048e8", f"{quality}.{fmt}", "quality.format: default · gray · bitonal  /  jpg · png · webp"),
    ]
    parts = "<span style='color:#adb5bd'>/</span>".join(
        f"<span title='{html.escape(t)}' style='color:{c};border-bottom:3px solid {c};padding-bottom:2px;cursor:help'>{html.escape(s)}</span>"
        for c, s, t in segs)
    legend = "".join(
        f"<span style='margin-right:14px;font-size:12px'><b style='color:{c}'>■</b> {html.escape(t.split(':')[0])}</span>"
        for c, s, t in segs)
    return (f"<div style='font:14px/2.2 ui-monospace,monospace;word-break:break-all;background:#f4f7fc;"
            f"border:1px solid #d5ddec;border-radius:8px;padding:10px 14px;color:#1d2433'>{parts}"
            f"<div style='font-family:system-ui;line-height:1.6'>{legend}</div></div>")

display(HTML(url_anatomy(region=f"{FULL_W//4},{FULL_H//4},{FULL_W//2},{FULL_H//4}", size="400,")))

## One image, many views: only the URL changes

Each tile below is a separate request to the same image server. Nothing was downloaded or edited locally.

In [ ]:
def gallery():
    w, h = FULL_W, FULL_H
    views = [
        ("Whole image, 220 px wide", "full", "220,", "0", "default"),
        ("Crop: centre half", f"{w//4},{h//4},{w//2},{h//2}", "220,", "0", "default"),
        ("Crop: top-left corner", f"0,0,{w//3},{h//4}", "220,", "0", "default"),
        ("Rotated 90°", "full", "!220,220", "90", "default"),
        ("Mirrored", "full", "220,", "!0", "default"),
        ("Greyscale", "full", "220,", "0", "gray"),
        ("Bitonal (black / white)", "full", "220,", "0", "bitonal"),
        ("Square crop", "square", "220,", "0", "default"),
        ("Tiny thumbnail, 20 %", "full", "pct:20", "0", "default"),
    ]
    tiles = ""
    for title, region, size, rot, q in views:
        path = f"{region}/{size}/{rot}/{q}.jpg"
        tiles += (f"<figure class='gt'><div class='gi'><img src='{service}/{path}' alt=''></div>"
                  f"<figcaption><b>{title}</b><code>…/{path}</code></figcaption></figure>")
    return """
<style>
.gg{display:grid;grid-template-columns:repeat(auto-fill,minmax(230px,1fr));gap:12px;font-family:system-ui,sans-serif}
.gt{margin:0;background:#f4f7fc;border:1px solid #d5ddec;border-radius:8px;overflow:hidden;color:#1d2433}
.gi{height:200px;display:flex;align-items:center;justify-content:center;background:#e9edf5}
.gi img{max-width:100%;max-height:200px}
.gt figcaption{padding:8px 10px;font-size:13px}
.gt code{display:block;margin-top:3px;font-size:10.5px;color:#5b6577;word-break:break-all}
</style><div class="gg">""" + tiles + "</div>"

display(HTML(gallery()))

## Your turn: ask for one region

The URL pattern lets you request any rectangle with `x,y,w,h` (pixels of the full-size image, origin top-left):

```python
Image(url=f"{service}/{x},{y},{w},{h}/full/0/default.jpg")
```

The next cell uses a region that fits whatever image was loaded. Change the four numbers and re-run. The tool further down will help you find good ones.

In [ ]:
x, y, w, h = FULL_W // 4, FULL_H // 4, FULL_W // 2, FULL_H // 4   # x,y,w,h in full-resolution pixels

Image(url=f"{service}/{x},{y},{w},{h}/full/0/default.jpg")

## Find coordinates by clicking, then annotate

Guessing pixel coordinates is tedious. The tool below loads the image from its IIIF service and lets you:

1. **Drag** a rectangle. The `x, y, w, h` fill in (you can also type them) and a live crop is fetched from the server.
2. **Type a label** and press *Add* to create an annotation.
3. **Copy** the result as a Python snippet, as an `ANNOTATIONS` list for the next cell, or as a standard **IIIF/W3C annotation** whose `target` is `canvas#xywh=…`.

> The canvas may use a different coordinate space than the full-resolution image (here it's smaller). The exported IIIF JSON converts for you. That is exactly the kind of detail the standard makes explicit.

In [ ]:
ANNOTATOR_HTML = r'''<!doctype html><meta charset="utf-8">
<style>
*{box-sizing:border-box}
body{margin:0;font:14px/1.4 system-ui,sans-serif;color:#1d2433;background:#fff}
.wrap{display:grid;grid-template-columns:minmax(260px,1.3fr) minmax(260px,1fr);gap:16px;padding:12px}
@media(max-width:700px){.wrap{grid-template-columns:1fr}}
h4{margin:0 0 6px;font-size:12px;letter-spacing:.06em;text-transform:uppercase;color:#5b6577}
.stage{position:relative;display:inline-block;line-height:0;cursor:crosshair;touch-action:none;user-select:none;border:1px solid #cfd6e4;border-radius:6px;overflow:hidden;background:#eef1f7}
.stage img{display:block;max-width:100%;max-height:520px;-webkit-user-drag:none}
.box{position:absolute;border:2px solid #e8590c;background:rgba(232,89,12,.18);pointer-events:none}
.ann{position:absolute;border:2px solid #1c7ed6;background:rgba(28,126,214,.12);pointer-events:none}
.ann span{position:absolute;left:-2px;top:-20px;background:#1c7ed6;color:#fff;font:11px/16px system-ui;padding:0 5px;border-radius:3px;white-space:nowrap}
.hint{font-size:12px;color:#5b6577;margin-top:6px}
.row{display:flex;gap:6px;margin:6px 0}
.row label{flex:1;font-size:11px;color:#5b6577}
.row input{width:100%;padding:4px 6px;border:1px solid #cfd6e4;border-radius:4px;font:13px ui-monospace,monospace}
input.txt{width:100%;padding:6px 8px;border:1px solid #cfd6e4;border-radius:4px;font:14px system-ui}
button{padding:6px 10px;border:1px solid #1c7ed6;background:#1c7ed6;color:#fff;border-radius:4px;cursor:pointer;font:13px system-ui}
button.ghost{background:#fff;color:#1c7ed6}
button:disabled{opacity:.4;cursor:default}
pre{margin:6px 0;padding:8px;background:#0f1729;color:#d6e4ff;border-radius:6px;font:12px/1.5 ui-monospace,monospace;white-space:pre-wrap;word-break:break-all;max-height:150px;overflow:auto}
.crop{min-height:90px;display:flex;align-items:center;justify-content:center;background:#eef1f7;border-radius:6px;border:1px dashed #cfd6e4;margin:6px 0}
.crop img{max-width:100%;max-height:200px}
ul{list-style:none;margin:6px 0;padding:0}
li{display:flex;justify-content:space-between;gap:8px;padding:4px 8px;margin-bottom:4px;background:#f1f5fb;border-left:3px solid #1c7ed6;border-radius:3px;font-size:13px}
li code{color:#5b6577;font-size:11px}
li a{color:#c92a2a;cursor:pointer;text-decoration:none}
.ok{color:#2b8a3e;font-size:12px;margin-left:6px}
</style>
<div class="wrap">
 <div>
  <h4>1 · Drag a rectangle on the image</h4>
  <div class="stage" id="stage"><img id="img" alt="" draggable="false"></div>
  <div class="hint" id="hint">Image is <b id="dim"></b> px at full resolution (from the service's <code>info.json</code>).</div>
 </div>
 <div>
  <h4>2 · Region in full-resolution pixels</h4>
  <div class="row">
   <label>x<input id="x" type="number" min="0"></label><label>y<input id="y" type="number" min="0"></label>
   <label>w<input id="w" type="number" min="1"></label><label>h<input id="h" type="number" min="1"></label>
  </div>
  <div class="crop" id="crop"><span class="hint">live crop appears here</span></div>
  <h4>3 · Annotate it</h4>
  <div class="row"><input class="txt" id="label" placeholder="What is in this region?"><button id="add" disabled>Add</button></div>
  <ul id="list"></ul>
  <h4>4 · Take it with you</h4>
  <div class="row">
   <button class="ghost" id="cpy">Copy Python snippet</button>
   <button class="ghost" id="cpl">Copy ANNOTATIONS list</button>
   <button class="ghost" id="cpj">Copy IIIF annotation JSON</button><span class="ok" id="ok"></span>
  </div>
  <pre id="out"></pre>
 </div>
</div>
<script>
const CFG = __CFG__;
const $ = id => document.getElementById(id);
const img = $('img'), stage = $('stage');
let natW = CFG.width, natH = CFG.height, sel = null, anns = [], drag = null;
img.src = CFG.service + '/full/!1000,1000/0/default.jpg';
$('dim').textContent = natW + ' × ' + natH;
const scale = () => natW / img.clientWidth;

function drawBox(el, r, s) {
  el.style.left = r.x / s + 'px'; el.style.top = r.y / s + 'px';
  el.style.width = r.w / s + 'px'; el.style.height = r.h / s + 'px';
}
function render() {
  stage.querySelectorAll('.box,.ann').forEach(n => n.remove());
  const s = scale();
  anns.forEach(a => {
    const d = document.createElement('div'); d.className = 'ann';
    drawBox(d, a, s); d.innerHTML = '<span></span>'; d.firstChild.textContent = a.label; stage.appendChild(d);
  });
  if (sel) { const b = document.createElement('div'); b.className = 'box'; drawBox(b, sel, s); stage.appendChild(b); }
  ['x','y','w','h'].forEach(k => { if (document.activeElement !== $(k)) $(k).value = sel ? sel[k] : ''; });
  $('add').disabled = !sel;
  $('list').innerHTML = '';
  anns.forEach((a, i) => {
    const li = document.createElement('li');
    li.innerHTML = '<span></span><span><code></code> <a title="remove">✕</a></span>';
    li.children[0].textContent = a.label; li.querySelector('code').textContent = `${a.x},${a.y},${a.w},${a.h}`;
    li.querySelector('a').onclick = () => { anns.splice(i, 1); render(); };
    $('list').appendChild(li);
  });
  output();
}
const clamp = (v, lo, hi) => Math.max(lo, Math.min(hi, v));
function pt(e) {
  const r = img.getBoundingClientRect(), s = scale();
  return { x: clamp((e.clientX - r.left) * s, 0, natW), y: clamp((e.clientY - r.top) * s, 0, natH) };
}
stage.addEventListener('pointerdown', e => { stage.setPointerCapture(e.pointerId); drag = pt(e); });
stage.addEventListener('pointermove', e => {
  if (!drag) return;
  const p = pt(e);
  sel = { x: Math.round(Math.min(drag.x, p.x)), y: Math.round(Math.min(drag.y, p.y)),
          w: Math.max(1, Math.round(Math.abs(p.x - drag.x))), h: Math.max(1, Math.round(Math.abs(p.y - drag.y))) };
  render();
});
stage.addEventListener('pointerup', () => { if (drag) { drag = null; preview(); } });
['x','y','w','h'].forEach(k => $(k).addEventListener('input', () => {
  const v = { x: +$('x').value, y: +$('y').value, w: +$('w').value, h: +$('h').value };
  if ([v.x,v.y,v.w,v.h].some(isNaN) || v.w < 1 || v.h < 1) return;
  v.x = clamp(v.x, 0, natW - 1); v.y = clamp(v.y, 0, natH - 1);
  v.w = Math.min(v.w, natW - v.x); v.h = Math.min(v.h, natH - v.y);
  sel = v; render(); preview();
}));
function preview() {
  if (!sel) return;
  $('crop').innerHTML = '';
  const c = new Image(); c.src = `${CFG.service}/${sel.x},${sel.y},${sel.w},${sel.h}/!400,200/0/default.jpg`;
  $('crop').appendChild(c);
}
$('add').onclick = () => {
  const label = $('label').value.trim() || 'untitled';
  anns.push({ ...sel, label }); $('label').value = ''; sel = null;
  $('crop').innerHTML = '<span class="hint">live crop appears here</span>'; render();
};
$('label').addEventListener('keydown', e => { if (e.key === 'Enter' && !$('add').disabled) $('add').click(); });

const q = s => JSON.stringify(s);
const snippet = () => sel
  ? `service = "${CFG.service}"\nImage(url=f"{service}/${sel.x},${sel.y},${sel.w},${sel.h}/full/0/default.jpg")`
  : '# drag a rectangle first';
const annList = () => 'ANNOTATIONS = [\n' + anns.map(a => `    {"label": ${q(a.label)}, "region": "${a.x},${a.y},${a.w},${a.h}"},`).join('\n') + '\n]';
function annJson() {
  const k = CFG.canvasWidth / natW;       // image pixels -> canvas units
  return JSON.stringify({
    '@context': 'http://iiif.io/api/presentation/3/context.json',
    id: CFG.canvasId + '/annotations', type: 'AnnotationPage',
    items: anns.map((a, i) => ({
      id: `${CFG.canvasId}/annotations/${i + 1}`, type: 'Annotation', motivation: 'commenting',
      body: { type: 'TextualBody', language: 'en', format: 'text/plain', value: a.label },
      target: `${CFG.canvasId}#xywh=${[a.x, a.y, a.w, a.h].map(v => Math.round(v * k)).join(',')}`
    }))
  }, null, 2);
}
function output() { $('out').textContent = snippet() + '\n\n' + (anns.length ? annList() : '# add annotations to see them here'); }
function copy(text) {
  const t = document.createElement('textarea'); t.value = text; document.body.appendChild(t); t.select();
  let ok = false; try { ok = document.execCommand('copy'); } catch (e) {}
  t.remove(); $('ok').textContent = ok ? 'copied ✓' : 'copy blocked: select the text below instead';
  if (!ok) $('out').textContent = text; setTimeout(() => $('ok').textContent = '', 2500);
}
$('cpy').onclick = () => copy(snippet());
$('cpl').onclick = () => copy(annList());
$('cpj').onclick = () => copy(annJson());
img.onload = render; window.addEventListener('resize', render); render();
</script>
'''

cfg = {"service": service, "width": FULL_W, "height": FULL_H,
       "canvasId": item["canvas_id"], "canvasWidth": item["canvas_w"]}
page = ANNOTATOR_HTML.replace("__CFG__", json.dumps(cfg))

display(HTML(f"<iframe srcdoc='{html.escape(page, quote=True)}' "
             "style='width:100%;height:680px;border:1px solid #d5ddec;border-radius:8px;background:#fff'></iframe>"))

## From annotations back to pixels

Paste the output of **Copy ANNOTATIONS list** over the example below. Each annotation becomes a crop fetched straight from the institution's server, with your label attached.

In [ ]:
ANNOTATIONS = [
    {"label": "example: top of the image", "region": f"0,0,{FULL_W},{FULL_H // 4}"},
    {"label": "example: centre", "region": f"{FULL_W // 3},{FULL_H // 3},{FULL_W // 3},{FULL_H // 3}"},
]

tiles = "".join(
    f"<figure style='margin:0;text-align:center;font:13px system-ui;color:#1d2433'>"
    f"<img src='{service}/{a['region']}/!300,300/0/default.jpg' style='max-height:220px;border:1px solid #d5ddec;border-radius:6px'>"
    f"<figcaption><b>{html.escape(a['label'])}</b><br><code>{a['region']}</code></figcaption></figure>"
    for a in ANNOTATIONS)
display(HTML(f"<div style='display:flex;gap:16px;flex-wrap:wrap'>{tiles}</div>"))

---

## Summary

1. A **manifest** gives you metadata, rights, and the address of an **image service**.
2. The **Image API** turns a URL into a region, size, rotation and colour mode of that image. No downloads, no editing software.
3. **Annotations** are just text plus a `canvas#xywh=` target, so they can be shared, merged and re-used across viewers.

**Caveat for your own research:** not every Europeana item has an IIIF image service. Many institutions only expose a static image URL. Check the `service` field in the manifest before relying on region requests.

### Useful Resources
- **IIIF**: https://iiif.io/
- **Image API 3.0 spec**: https://iiif.io/api/image/3.0/
- **Presentation API 3.0 spec**: https://iiif.io/api/presentation/3.0/
- **IIIF Cookbook (recipes)**: https://iiif.io/api/cookbook/

### Next
Continue with **02_download_and_clip_search.ipynb**.
